In [1]:
# 3.10.0

In [2]:
import numpy as np 
import pandas as pd
import os

In [3]:
link='/Users/Tysia/Desktop/Edukacja/Informatyka/Magisterka/ml-1m'
rating_data=os.path.join(link,'ratings.dat')
movie_data=os.path.join(link,'movies.dat')
user_data=os.path.join(link,'users.dat')

ratings=pd.read_csv(
  rating_data,
  sep='::',
  engine='python',
  names=['user','item','rating','timestamp'],
  encoding='latin-1'   
)

movies=pd.read_csv(
    movie_data, 
    sep='::',
    engine='python',
    names=['item', 'title', 'genres'],
    encoding='latin-1'
)
users = pd.read_csv(
    user_data, 
    sep='::',
    engine='python',
    names=['user', 'gender', 'age', 'occupation', 'zipcode'],
    encoding='latin-1'
)

In [ ]:
print(ratings.head())
print(movies.head())
print(users.head())

   user_id  movie_id  rating  timestamp
0        1      1193       5  978300760
1        1       661       3  978302109
2        1       914       3  978301968
3        1      3408       4  978300275
4        1      2355       5  978824291
   movie_id                               title                        genres
0         1                    Toy Story (1995)   Animation|Children's|Comedy
1         2                      Jumanji (1995)  Adventure|Children's|Fantasy
2         3             Grumpier Old Men (1995)                Comedy|Romance
3         4            Waiting to Exhale (1995)                  Comedy|Drama
4         5  Father of the Bride Part II (1995)                        Comedy
   user_id gender  age  occupation zipcode
0        1      F    1          10   48067
1        2      M   56          16   70072
2        3      M   25          15   55117
3        4      M   45           7   02460
4        5      M   25          20   55455


In [5]:
from sklearn.model_selection import KFold, train_test_split
from lenskit.algorithms import item_knn, user_knn, basic, als, implicit
from lenskit.algorithms.implicit import BPR
from lenskit import batch, metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error

c:\Users\Tysia\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
train, test = train_test_split(ratings, test_size=0.2, random_state=42)

In [ ]:
# nnbrs - liczba sąsiadów (k)
# center - centrowanie oceny, odejmowanie średniej
# aggregate - jak łączyć oceny sąsiadów
iknn = item_knn.ItemItem(nnbrs=20, center=True, aggregate='weighted-average')
iknn.fit(train)

Numba is using threading layer omp - consider TBB
found 1 potential runtime problems - see https://boi.st/lkpy-perf
c:\Users\Tysia\AppData\Local\Programs\Python\Python310\lib\site-packages\lenskit\algorithms\item_knn.py:119: NumbaTypeSafetyWarning: unsafe cast from uint64 to int64. Precision may be lost.
  b = blocks[bi]


In [28]:
predictions = batch.predict(iknn, test).dropna(subset=['prediction'])
rmse = np.sqrt(mean_squared_error(predictions['rating'], predictions['prediction']))
mae = mean_absolute_error(predictions['rating'], predictions['prediction'])
rmse, mae

(0.8549444773912852, 0.6684801920729448)

In [ ]:
# nnbrs - liczba sąsiadów, brani są pod uwagę najbardziej podobni użytkownicy
# center - normalizowanie oceny, odejmowanie średniej oceny
uknn = user_knn.UserUser(nnbrs=20) # center=True
uknn.fit(train)

Numba is using threading layer omp - consider TBB
found 1 potential runtime problems - see https://boi.st/lkpy-perf


In [ ]:
#with center=False
predictions = batch.predict(uknn, test).dropna(subset=['prediction'])
rmse = np.sqrt(mean_squared_error(predictions['rating'], predictions['prediction']))
mae = mean_absolute_error(predictions['rating'], predictions['prediction'])
rmse, mae

(0.8985434388810448, 0.702190268497957)

In [ ]:
#with center=True
predictions = batch.predict(uknn, test).dropna(subset=['prediction'])
rmse = np.sqrt(mean_squared_error(predictions['rating'], predictions['prediction']))
mae = mean_absolute_error(predictions['rating'], predictions['prediction'])
rmse, mae

(0.8985434388810448, 0.702190268497957)

In [ ]:
# MF
# features - liczba ukrytych cech  -> odpowiednik n_factors/k
# iterations - liczba epok uczenia
# reg - współczynnik regularyzacji, zapobiega przeuczeniu
mf = als.BiasedMF(features=10, iterations=30, reg=0.015)
mf.fit(train)

In [41]:
predictions = batch.predict(mf, test).dropna(subset=['prediction'])
rmse = np.sqrt(mean_squared_error(predictions['rating'], predictions['prediction']))
mae = mean_absolute_error(predictions['rating'], predictions['prediction'])
rmse, mae

(0.8946096386351764, 0.6927031326455165)

In [35]:
# BPR MF
bpr = BPR(factors=10, iterations=30, learning_rate=0.05, regularization=0.0025, random_state=42)
bpr.fit(train)


100%|██████████| 30/30 [00:04<00:00,  6.15it/s, train_auc=81.61%, skipped=21.90%]


In [36]:
from lenskit import topn
from lenskit.metrics.topn import ndcg, precision, recall

In [ ]:
# trzeba wygenerować listę top-n filmów dla każdego użytkownika z testu i wybrać unikalnych użytkowników ze zbioru testowego
test_users = test['user'].unique()

# generowanie 5 rekomendacji dla każdego 
recs = batch.recommend(bpr, test_users, 5)

# połączenie rekomendacji(recs) z prawdą (test_data), aby sprawdzić trafność
rla = topn.RecListAnalysis() # narzędzie do analizy rankingów
rla.add_metric(ndcg)
rla.add_metric(precision)
rla.add_metric(recall)
results = rla.compute(recs, test)
results.head

c:\Users\Tysia\AppData\Local\Programs\Python\Python310\lib\site-packages\lenskit\metrics\topn.py:100: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  scores['ngood'].fillna(0, inplace=True)


<bound method NDFrame.head of       nrecs      ndcg  precision    recall
user                                      
5412      5  0.225137        1.0  0.078125
5440      5  0.000000        0.0  0.000000
368       5  0.000000        0.0  0.000000
425       5  0.116015        0.2  0.030303
4942      5  0.131144        0.6  0.050847
...     ...       ...        ...       ...
642       5  0.000000        0.0  0.000000
2982      5  0.000000        0.0  0.000000
4332      5  0.000000        0.0  0.000000
66        5  0.000000        0.0  0.000000
4065      5  0.000000        0.0  0.000000

[6038 rows x 4 columns]>

In [39]:
precision = results['precision'].mean()
recall = results['recall'].mean()
ndcg = results['ndcg'].mean()

precision, recall, ndcg

(0.17545544882411396, 0.03612908625583849, 0.06261369728995785)

In [20]:
#mostpop
mostpop = basic.Popular()

mostpop.fit(train)

In [21]:
test_users = test['user'].unique()
recs = batch.recommend(mostpop, test_users, 5)

rla = topn.RecListAnalysis()
rla.add_metric(recall)
rla.add_metric(precision)
rla.add_metric(ndcg)

results = rla.compute(recs, test)

results

precision = results['precision'].mean()
recall = results['recall'].mean()
ndcg = results['ndcg'].mean()

precision, recall, ndcg

c:\Users\Tysia\AppData\Local\Programs\Python\Python310\lib\site-packages\lenskit\metrics\topn.py:100: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  scores['ngood'].fillna(0, inplace=True)


(0.2104670420669096, 0.039663198764016694, 0.07731520553236805)